In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from analytics.forecasting.base import SimpleForecaster
from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_BASELINE,
    ARTIFACTS_DIR,
    TICKERS,
)
SPAN = 20

In [ ]:
# Load pool: all tickers stacked into one DataFrame
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close
0,2021-03-01,AAPL,127.790001
1,2021-03-02,AAPL,125.120003
2,2021-03-03,AAPL,122.059998
3,2021-03-04,AAPL,120.129997
4,2021-03-05,AAPL,121.419998
5,2021-03-08,AAPL,116.360001
6,2021-03-09,AAPL,121.089996
7,2021-03-10,AAPL,119.980003
8,2021-03-11,AAPL,121.959999
9,2021-03-12,AAPL,121.029999


In [6]:
stacked.describe()

,timestamp,close
count,12560,12560.000000
mean,2023-08-27 08:25:36.305000,194.495601
min,2021-03-01 00:00:00,11.227000
25%,2022-05-25 18:00:00,109.129997
50%,2023-08-26 12:00:00,158.250000
75%,2024-11-22 18:00:00,234.557499
max,2026-02-27 00:00:00,695.489990
std,NaN,138.142456


In [3]:
# Train once on pooled data: average close by date across assets (before 60-day test), fit one EWM
pooled_train = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_BASELINE)
if not pooled_train.empty:
    avg_series = pooled_train.groupby("timestamp")["close"].mean().sort_index()
    if len(avg_series) >= MIN_TRAIN_BASELINE:
        global_baseline_model = SimpleForecaster(span=SPAN, confidence_level=0.95)
        global_baseline_model.fit(avg_series)
        global_baseline_last = float(avg_series.iloc[-1])
    else:
        global_baseline_model = None
        global_baseline_last = None
else:
    global_baseline_model = None
    global_baseline_last = None


def get_forecast_baseline(context: pd.Series, horizon: int):
    if global_baseline_model is None or global_baseline_last is None:
        return []
    fc = global_baseline_model.forecast(periods=horizon)
    point = (fc.get("point_forecast") or []) if fc else []
    if not point:
        return []
    scale = float(context.iloc[-1]) / global_baseline_last
    return [float(x) * scale for x in point]


# Backtest: 60-day test window, rolling by 1 week; predict only (global model trained above)
model_name = "baseline"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    close_ser = grp.set_index("timestamp")["close"]
    if isinstance(close_ser, pd.DataFrame):
        close_ser = close_ser.iloc[:, 0] if close_ser.shape[1] == 1 else close_ser[sym] if sym in close_ser.columns else close_ser.iloc[:, 0]
    prices = close_ser.astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_TRAIN_BASELINE:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_TRAIN_BASELINE,
        get_forecast_fn=get_forecast_baseline,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_baseline = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_baseline.groupby("symbol").size() if not pred_baseline.empty else "No predictions.")
pred_baseline.head()

symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,278.997445,0,AAPL
1,2025-12-03,284.149994,278.997445,0,AAPL
2,2025-12-04,280.700012,278.997445,0,AAPL
3,2025-12-05,278.779999,278.997445,0,AAPL
4,2025-12-08,277.890015,278.997445,0,AAPL


In [4]:
# Metrics per symbol and overall: averaged over rolling mini-windows (MAE, RMSE, MAPE_%)
metrics_rows = []
for sym in pred_baseline["symbol"].unique():
    sub = pred_baseline[pred_baseline["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_baseline)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_baseline_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_baseline_pool.parquet")

       model   symbol        MAE       RMSE    MAPE_%
0   baseline     AAPL   9.698705  11.500091  3.673588
1   baseline     MSFT  18.451262  22.183884  4.267448
2   baseline    GOOGL  11.865465  13.372779  3.653935
3   baseline     AMZN  10.424299  12.661312  4.608247
4   baseline      JPM  12.698524  14.034067  4.001225
5   baseline      JNJ  11.604979  13.120494  5.123779
6   baseline      WMT   5.221177   6.209763  4.295913
7   baseline      SPY  12.556049  13.331984  1.820749
8   baseline      XOM   9.593406  10.802033  7.022775
9   baseline     NVDA   5.846683   6.514533  3.144116
10  baseline  overall  10.796055  13.865619  4.161178
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_baseline_pool.parquet
